# Obtaining numerical solutions

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/07-solving-the-system.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Up to now we have built models, run them, and plotted trajectories. This
chapter makes that process concrete: what it means to *solve* the system of
flows, how a manual Euler step relates to `solver="euler"`, and why an
adaptive Runge–Kutta method (Dopri5) is the better default for stiff or
suddenly changing rates.


## Ordinary differential equations

For frequency-dependent transmission the SIR system is

$$
\frac{dS}{dt}=-\frac{\beta S I}{N},\quad
\frac{dI}{dt}=\frac{\beta S I}{N}-\gamma I,\quad
\frac{dR}{dt}=\gamma I
$$

summer4 expresses the same structure with compartments and named flows rather
than writing the ODEs by hand. The vector field is still available for a
manual step if you want to see the arithmetic.


In [ ]:
from typing import Any, NamedTuple

import jax.numpy as jnp
import numpy as np

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
    derived_refs,
)

state = Property("state", ("susceptible", "infectious", "recovered"))
pmap = PropertyMap.from_property(state)
I_IDX = int(pmap.select(state["infectious"])[0])


class SirParams(NamedTuple):
    contact_rate: float
    recovery: float
    foi: float = 0.0


refs = derived_refs(SirParams)


def derived_fn(params: SirParams, *, y: object, t: object) -> SirParams:
    del t
    data = y.data if isinstance(y, PropertyData) else y
    data = jnp.asarray(data)
    n_infect = data[I_IDX]
    n_pop = jnp.sum(data)
    foi = params.contact_rate * n_infect / n_pop
    return SirParams(params.contact_rate, params.recovery, foi)


def build_sir() -> Any:
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "infection",
            state["susceptible"],
            state["infectious"],
            refs.foi,
        )
    )
    model.add_flow(
        TransitionFlow(
            "recovery",
            state["infectious"],
            state["recovered"],
            refs.recovery,
        )
    )
    return model.compile(derived_fn=derived_fn)


model_config = {"population": 1000.0, "seed": 10.0, "end_time": 20.0, "time_step": 1.0}
parameters = SirParams(contact_rate=1.0, recovery=0.333)
cm = build_sir()
y0 = PropertyData.wrap(
    pmap,
    np.array(
        [
            model_config["population"] - model_config["seed"],
            model_config["seed"],
            0.0,
        ]
    ),
)
plan = SavePlan(requests={"compartments": SaveRequest(Compartments())})


## Manual Euler evaluation

The loop below is what a fixed-step Euler solver does: evaluate the flow rates
at the current state, then advance `y <- y + dt * dy`.


In [ ]:
time_period = model_config["end_time"]
num_steps = int(time_period / model_config["time_step"]) + 1
times = np.linspace(0.0, model_config["end_time"], num=num_steps)
manual = np.zeros((num_steps, 3))
manual[0] = np.asarray(y0.data)
dt = model_config["time_step"]
for t_idx in range(1, num_steps):
    t = times[t_idx - 1]
    dy = cm.vector_field(t, PropertyData.wrap(pmap, manual[t_idx - 1]), parameters)
    dy_arr = np.asarray(dy.data if isinstance(dy, PropertyData) else dy)
    manual[t_idx] = manual[t_idx - 1] + dy_arr * dt


## summer4 Euler matches the manual loop

In [ ]:
euler = cm.run(
    parameters,
    y0,
    t0=0.0,
    steps=int(model_config["end_time"] / model_config["time_step"]),
    dt=model_config["time_step"],
    save=plan,
    solver="euler",
)
euler_y = np.asarray(euler["compartments"].values.data)
np.testing.assert_allclose(euler_y, manual, rtol=1e-4, atol=1e-4)
assert float(np.max(np.abs(euler_y - manual))) < 1e-3


## Choice of solvers

Euler is kept as a reference stepper (spreadsheet models, teaching). For
policy work, use an adaptive Runge–Kutta method — here Dopri5 via diffrax.
The next cell compares step counts on the smooth baseline epidemic.


In [ ]:
dopri = cm.run(
    parameters,
    y0,
    t0=0.0,
    t1=model_config["end_time"],
    dt=model_config["time_step"],
    save=plan,
    solver="dopri5",
    rtol=1e-6,
    atol=1e-8,
)
assert dopri.solver is not None
assert int(dopri.solver.result_code) == 0
assert dopri.solver.message  # non-empty host-side status
dopri_y = np.asarray(dopri["compartments"].values.data)
assert np.all(np.isfinite(dopri_y))
assert float(np.min(dopri_y)) >= -1e-6
# Adaptive Dopri5 chooses its own steps; coarse Euler used 20 unit steps.
assert int(dopri.solver.num_steps) != int(euler.solver.num_steps)
print("euler steps:", int(euler.solver.num_steps), "dopri5 steps:", int(dopri.solver.num_steps))
print("final euler:", euler_y[-1], "final dopri5:", dopri_y[-1])


## A sudden parameter spike

When the contact rate jumps for a short window, a coarse Euler step can drive
susceptibles negative by extrapolating the peak force of infection across the
whole interval. Dopri5 adapts and keeps compartments non-negative.


In [ ]:
class SpikeParams(NamedTuple):
    contact_rate: float
    recovery: float
    contact_rate_step_start_time: float
    contact_rate_step_duration: float
    contact_rate_step_value: float
    foi: float = 0.0


spike_refs = derived_refs(SpikeParams)


def spike_derived(params: SpikeParams, *, y: object, t: object) -> SpikeParams:
    data = y.data if isinstance(y, PropertyData) else y
    data = jnp.asarray(data)
    n_infect = data[I_IDX]
    n_pop = jnp.sum(data)
    time = jnp.asarray(t)
    offset = time - params.contact_rate_step_start_time
    beta = jnp.where(
        (offset > 0.0) & (offset < params.contact_rate_step_duration),
        params.contact_rate_step_value,
        params.contact_rate,
    )
    return SpikeParams(
        params.contact_rate,
        params.recovery,
        params.contact_rate_step_start_time,
        params.contact_rate_step_duration,
        params.contact_rate_step_value,
        beta * n_infect / n_pop,
    )


def build_spike() -> Any:
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow("infection", state["susceptible"], state["infectious"], spike_refs.foi)
    )
    model.add_flow(
        TransitionFlow("recovery", state["infectious"], state["recovered"], spike_refs.recovery)
    )
    return model.compile(derived_fn=spike_derived)


spike_params = SpikeParams(
    contact_rate=1.0,
    recovery=0.333,
    contact_rate_step_start_time=10.0,
    contact_rate_step_duration=2.0,
    contact_rate_step_value=10.0,
)
cm_spike = build_spike()
euler_spike = cm_spike.run(
    spike_params, y0, t0=0.0, steps=20, dt=1.0, save=plan, solver="euler"
)
dopri_spike = cm_spike.run(
    spike_params,
    y0,
    t0=0.0,
    t1=20.0,
    dt=1.0,
    save=plan,
    solver="dopri5",
    rtol=1e-6,
    atol=1e-8,
)
euler_s = np.asarray(euler_spike["compartments"].values.data)[:, 0]
dopri_s = np.asarray(dopri_spike["compartments"].values.data)[:, 0]
assert float(np.min(euler_s)) < 0.0  # coarse Euler overshoots
assert float(np.min(dopri_s)) >= -1e-6
print("min S euler:", float(np.min(euler_s)), "min S dopri5:", float(np.min(dopri_s)))
